# 08. 三分支融合（硬件友好量化后）+ 资源估算

最后一个 notebook：用 Context-HW（notebook 02）、Hybrid-HW（notebook 04）、
Delay-SNN（notebook 05/06）重新做三分支融合，复现量化后 **91.11%** 的测试集结果，
然后做资源估算，对应教程 08 章的方法论。

**关于 Delay 分支怎么参与融合，有一点需要说清楚**：融合需要的是校准过的概率分布
（配合温度缩放的 softmax），而 notebook 06 里做出来的完整定点版本
（`FixedDelaySNN`/`fixed_inference`）只输出**每类原始脉冲计数**这种更贴近硬件真实
输出的整数量，不直接产出适合概率加权融合用的分布。真实项目的融合脚本
（`evaluate_ensemble.py`/`evaluate_hw_ensemble.py`）在这两种"量化后三分支融合"和
"FP32 三分支融合"里，**Delay 分支用的都是同一个 FP32 概率模型**——量化只发生在
Delay 分支单独评估、单独上板的场景（notebook 06），融合这一层没有变。这里如实复现
这个设计，不做美化。

In [ ]:
import sys, importlib.util
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path("training/semg_snn_90_loop")
DELAY_PROJECT_ROOT = Path("training/semg_snn_fpga_reproduction")
sys.path.insert(0, str(PROJECT_ROOT))

from train import EMGDataset
from hw_model import HWClassAdaptiveContextSNN, HWHybridSNN
from hw_fixed_reference import HWFixedContext, HWFixedHybrid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NB_RUNS = PROJECT_ROOT / "runs_notebook"
WEIGHTS_DIR = NB_RUNS / "weights_hw"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

def resolve(nb_path: Path, fallback: Path) -> Path:
    return nb_path if nb_path.exists() else fallback

QAT_CONTEXT_CHECKPOINT = resolve(
    NB_RUNS / "hw_context23_qat_nb" / "best.pt",
    PROJECT_ROOT / "runs" / "hw_context23_qat_v1_affinefix" / "best.pt",
)
QAT_HYBRID_CHECKPOINT = resolve(
    NB_RUNS / "hw_hybrid_qat_nb" / "best.pt",
    PROJECT_ROOT / "runs" / "hw_hybrid_qat_v1_affinefix" / "best.pt",
)
DELAY_CHECKPOINT = resolve(
    DELAY_PROJECT_ROOT / "runs_notebook" / "delay62_finetune_nb" / "best.pt",
    DELAY_PROJECT_ROOT / "runs" / "delay62_finetune" / "best.pt",
)
print("Context-HW:", QAT_CONTEXT_CHECKPOINT)
print("Hybrid-HW: ", QAT_HYBRID_CHECKPOINT)
print("Delay:     ", DELAY_CHECKPOINT)

## 1. 量化后三分支融合

方法和 notebook 07 一样：验证集单纯形网格搜索融合权重 + Rest 偏置校准。
真实项目结果里量化后的融合权重（0.5/0.3/0.2）和 FP32 时的（0.5/0.4/0.1）不完全一样——
量化给三个分支的相对置信度带来了轻微变化，重新搜索权重能补偿这一点。

In [ ]:
@torch.no_grad()
def predict_delay(split: str, checkpoint: Path, temperature: float = 0.05):
    spec = importlib.util.spec_from_file_location("paper_snn_model", DELAY_PROJECT_ROOT / "model.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    state = torch.load(checkpoint, map_location=device, weights_only=False)
    saved_args = state.get("args", {})
    model = module.PaperSNNWithDelays(
        decay=saved_args.get("decay", 0.9), threshold=saved_args.get("threshold", 1.0),
        max_delay=saved_args.get("max_delay", 62), initial_delay=saved_args.get("initial_delay", 1.0),
    ).to(device)
    model.load_state_dict(state["model"])
    model.eval()
    source = np.load(DELAY_PROJECT_ROOT / "data" / "processed" / f"{split}.npz")
    x = torch.from_numpy(source["x"].astype(np.float32))
    y = source["y"].astype(np.int64)
    outputs = []
    for (batch,) in DataLoader(TensorDataset(x), 256, num_workers=4, pin_memory=True):
        spikes, _ = model(batch.to(device))
        outputs.append((spikes.mean(1) / temperature).softmax(1).cpu().numpy())
    return np.concatenate(outputs), y


@torch.no_grad()
def predict_context_hw(split: str, checkpoint: Path):
    dataset = EMGDataset(
        PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz",
        False, context=23, continuous_context=True, stream_context=True,
    )
    model = HWClassAdaptiveContextSNN(dataset.features.shape[1]).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=False)["model"])
    model.eval()
    outputs = []
    for f, raw, _, subject in DataLoader(dataset, 512, num_workers=4, pin_memory=True):
        logits, _ = model(f.to(device), raw.to(device), subject.to(device))
        outputs.append(logits.softmax(1).cpu().numpy())
    return np.concatenate(outputs), dataset.y


@torch.no_grad()
def predict_hybrid_hw(split: str, checkpoint: Path):
    dataset = EMGDataset(
        PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz", False, context=1
    )
    model = HWHybridSNN(dataset.features.shape[1]).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=False)["model"])
    model.eval()
    outputs = []
    for f, raw, _, subject in DataLoader(dataset, 512, num_workers=4, pin_memory=True):
        logits, _ = model(f.to(device), raw.to(device), subject.to(device))
        outputs.append(logits.softmax(1).cpu().numpy())
    return np.concatenate(outputs), dataset.y


def score(y, probability):
    p = probability.argmax(1)
    return {
        "accuracy": accuracy_score(y, p),
        "macro_f1": f1_score(y, p, average="macro"),
        "gesture_accuracy": float(np.mean(p[y != 0] == y[y != 0])),
    }

probabilities, labels = {"val": [], "test": []}, {}
for split in probabilities:
    p_context, y1 = predict_context_hw(split, QAT_CONTEXT_CHECKPOINT)
    p_hybrid, y2 = predict_hybrid_hw(split, QAT_HYBRID_CHECKPOINT)
    p_delay, y3 = predict_delay(split, DELAY_CHECKPOINT)
    assert np.array_equal(y1, y2) and np.array_equal(y1, y3), f"{split} 三个分支的标签顺序没对齐"
    probabilities[split] = [p_context, p_hybrid, p_delay]
    labels[split] = y1
    print(f"{split}: context={score(y1,p_context)} hybrid={score(y2,p_hybrid)} delay={score(y3,p_delay)}")

candidates = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(11) for b in range(11 - a)]
rows = [{"weights": w, **score(labels["val"], sum(wi * p for wi, p in zip(w, probabilities["val"])))}
        for w in candidates]
best = max(rows, key=lambda r: (r["accuracy"], r["macro_f1"]))
print("\n验证集选出的融合权重 (context, hybrid, delay):", best["weights"])

test_probability = sum(w * p for w, p in zip(best["weights"], probabilities["test"]))
print("测试集(未校准):", score(labels["test"], test_probability))

val_probability = sum(w * p for w, p in zip(best["weights"], probabilities["val"]))
val_log_probability = np.log(np.clip(val_probability, 1e-8, 1.0))
bias_rows = []
for rest_bias in np.linspace(-1.2, 0.4, 81):
    calibrated = val_log_probability.copy()
    calibrated[:, 0] += rest_bias
    bias_rows.append({"rest_logit_bias": float(rest_bias), **score(labels["val"], calibrated)})
best_bias = max(bias_rows, key=lambda r: (r["accuracy"], r["macro_f1"]))

test_log_probability = np.log(np.clip(test_probability, 1e-8, 1.0))
test_log_probability[:, 0] += best_bias["rest_logit_bias"]
final_metrics = score(labels["test"], test_log_probability)
print(f"\n=== 量化后三分支融合 + 校准（Rest bias={best_bias['rest_logit_bias']:.2f}）最终测试集结果 ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")
print("\n参考值(真实项目 hw_three_branch_fusion_metrics_affinefix.json):")
print("  accuracy=0.9111  macro_f1=0.8240  gesture_accuracy=0.7941")
print("\n对照 FP32 基线(notebook 07): accuracy=0.9110  macro_f1=0.8235  gesture_accuracy=0.7977")
print("=> 量化(INT4 权重 + Q8.8 激活 + 硬件友好算子替换)没有明显掉点。")

## 2. 导出定点权重（Context + Hybrid）

复现 notebook 02/04 第 6/4 节的导出逻辑，产出可以拿去和资源估算配合使用的
`.npz` 文件。这里不重复交叉验证（已经在各自的 notebook 里做过），只做导出。

In [ ]:
def export_context_fixed(state: dict, weight_bits: int, out_path: Path) -> None:
    arrays = {}
    def add_linear(prefix, weight_key, bias_key, bits):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 2 ** (bits - 1) - 1
        amax = np.maximum(np.abs(weight).max(axis=tuple(range(1, weight.ndim)), keepdims=True), 1e-8)
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = scale.reshape(-1).astype(np.float32)
        arrays[f"{prefix}_bias"] = bias
    def add_affine(prefix, weight_key, bias_key):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 127
        amax = float(np.maximum(np.abs(weight).max(), 1e-8))
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = np.float32(scale)
        arrays[f"{prefix}_bias"] = bias
    add_linear("enc_linear", "enc_linear.weight", "enc_linear.bias", weight_bits)
    add_affine("enc_affine", "enc_affine.weight", "enc_affine.bias")
    add_linear("fc2", "fc2.weight", "fc2.bias", weight_bits)
    add_affine("norm2", "norm2.weight", "norm2.bias")
    add_linear("out", "out.weight", "out.bias", weight_bits)
    np.savez(out_path, **arrays)
    print(f"wrote {out_path} ({sum(a.nbytes for a in arrays.values())/1024:.1f} KiB)")


def export_hybrid_fixed(state: dict, weight_bits: int, out_path: Path) -> None:
    arrays = {}
    def add_linear(prefix, weight_key, bias_key, bits):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 2 ** (bits - 1) - 1
        amax = np.maximum(np.abs(weight).max(axis=tuple(range(1, weight.ndim)), keepdims=True), 1e-8)
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = scale.reshape(-1).astype(np.float32)
        arrays[f"{prefix}_bias"] = bias
    def add_affine(prefix, weight_key, bias_key):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 127
        amax = float(np.maximum(np.abs(weight).max(), 1e-8))
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = np.float32(scale)
        arrays[f"{prefix}_bias"] = bias
    add_linear("feature_linear", "feature_linear.weight", "feature_linear.bias", weight_bits)
    add_affine("feature_affine", "feature_affine.weight", "feature_affine.bias")
    add_linear("conv1", "conv.conv1.weight", "conv.conv1.bias", weight_bits)
    add_linear("conv2", "conv.conv2.weight", "conv.conv2.bias", weight_bits)
    add_linear("q", "conv.q.weight", "conv.q.bias", weight_bits)
    add_linear("k", "conv.k.weight", "conv.k.bias", weight_bits)
    add_linear("v", "conv.v.weight", "conv.v.bias", weight_bits)
    add_linear("fuse_linear", "fuse_linear.weight", "fuse_linear.bias", weight_bits)
    add_affine("fuse_affine", "fuse_affine.weight", "fuse_affine.bias")
    add_linear("out", "out.weight", "out.bias", weight_bits)
    np.savez(out_path, **arrays)
    print(f"wrote {out_path} ({sum(a.nbytes for a in arrays.values())/1024:.1f} KiB)")


context_hw_state = torch.load(QAT_CONTEXT_CHECKPOINT, map_location=device, weights_only=False)["model"]
hybrid_hw_state = torch.load(QAT_HYBRID_CHECKPOINT, map_location=device, weights_only=False)["model"]
export_context_fixed(context_hw_state, weight_bits=4, out_path=WEIGHTS_DIR / "hw_context_fixed_nb.npz")
export_hybrid_fixed(hybrid_hw_state, weight_bits=4, out_path=WEIGHTS_DIR / "hw_hybrid_fixed_nb.npz")

## 3. 算子清单与资源估算(教程 08 章方法论)

先确认没有遗漏的昂贵浮点算子，再算权重占用——这里要小心一个坑（教程 08.2 节）：
`.npz` 里 INT4 codes 用 `int8` 容器存，每个值浪费一半空间；真实硬件会把两个 INT4
值紧凑打包进一个字节，资源估算要用打包后的字节数，不能直接用 npz 磁盘字节数。

In [ ]:
print("算子清单:")
print(f"{'原始算子':20s}{'替换方案':30s}")
for op, replacement in [
    ("GELU (erf)", "ReLU6"),
    ("LayerNorm", "HWAffine(逐通道仿射,可折叠)"),
    ("BatchNorm(推理态)", "折叠进前一层卷积权重"),
    ("Softmax", "留在 host 端,FPGA 只输出 logits"),
    ("Jaccard 除法", "整数索引查找表(101 项)"),
]:
    print(f"{op:20s}{replacement:30s}")


def hardware_realistic_bytes(npz_path: Path) -> tuple[float, float]:
    """返回 (npz 磁盘字节数, 假设 INT4 codes 真正 4-bit 紧凑打包后的字节数)。"""
    archive = np.load(npz_path)
    raw_bytes = sum(archive[k].nbytes for k in archive.files)
    packed_bytes = 0
    for k in archive.files:
        arr = archive[k]
        if k.endswith("_codes") and "affine" not in k:  # INT4 codes(HWAffine 是 INT8,不打包)
            packed_bytes += arr.nbytes / 2
        else:
            packed_bytes += arr.nbytes
    return raw_bytes, packed_bytes

context_raw, context_packed = hardware_realistic_bytes(WEIGHTS_DIR / "hw_context_fixed_nb.npz")
hybrid_raw, hybrid_packed = hardware_realistic_bytes(WEIGHTS_DIR / "hw_hybrid_fixed_nb.npz")
BOARD_BRAM_KIB = 607.5  # Nexys4 DDR (XC7A100T) 片上 BRAM 总容量,查芯片数据手册

print(f"\n{'':10s}{'npz 磁盘字节数':>16s}{'硬件 4-bit 紧凑打包':>20s}")
print(f"{'Context':10s}{context_raw/1024:14.1f} KiB{context_packed/1024:18.1f} KiB")
print(f"{'Hybrid':10s}{hybrid_raw/1024:14.1f} KiB{hybrid_packed/1024:18.1f} KiB")
packed_total_kib = (context_packed + hybrid_packed) / 1024
print(f"\nContext+Hybrid 硬件紧凑打包后合计: {packed_total_kib:.1f} KiB  /  板卡 BRAM 总量 {BOARD_BRAM_KIB} KiB "
      f"= {packed_total_kib/BOARD_BRAM_KIB*100:.1f}%")
print("\nDelay-SNN 分支不需要估算——它已经真实综合、烧录、测过硬件数字(notebook 06 第5节):")
print("  LUT 7.6%  /  BRAM 7 块  /  DSP 0  /  100MHz")
print("\nContext/Hybrid 的 LUT/DSP/FF 具体占用需要真正跑 Vitis HLS csynth + Vivado 才能给出确切数字，")
print("这里不编造百分比——完整讨论见 HW_QAT_FPGA_REPORT.md 和教程 08 章。")

## 小结

| | FP32 基线 (notebook 07) | HW-QAT 量化后 |
|---|---:|---:|
| 测试集 accuracy | 91.10% | 91.11% |
| 测试集 macro-F1 | 0.8235 | 0.8240 |
| 测试集 gesture-accuracy | 79.77% | 79.41% |

从浮点到 INT4 权重 + Q8.8 定点激活 + 全套硬件友好算子替换（Context/Hybrid），加上
更简单的训练后量化（Delay-SNN），**三分支融合的精度基本没有损失**——前提是量化过程
用了完整的方法（该用 QAT 的地方用 QAT，该用 PTQ 的地方用 PTQ），并且每一处算子替换
都经过 numpy 参考实现的位对位验证。

## 课程地图回顾

8 个 notebook 覆盖了三个结构不同的分支各自的训练与量化，加上两次融合：

```
01 Context 训练 ──→ 02 Context 量化 ─┐
03 Hybrid  训练 ──→ 04 Hybrid  量化 ─┼─→ 07 FP32 融合 (91.10%)
05 Delay   训练 ──→ 06 Delay   量化 ─┤    08 HW-QAT 融合 (91.11%)
                                     ┘
```

如果你是照着这套流程学别的模型/别的硬件目标，配套的 md 教程
[00-README.md](../00-README.md) 有更抽象、可移植到任意合成数据场景的版本，
可以作为讲给别人听时的教材；这 8 个 notebook 则是"这套方法在一个真实项目里
具体长什么样"的完整案例——包括它诚实的部分（Delay 分支量化确实掉了点精度，
Context/Hybrid 的 LUT/DSP 数字确实没法在这台机器上给出确切值）。